# Event-MAE on CIFAR10-DVS

Masked event-image modeling, MEM-style (Klenk et al., WACV 2024). **Reconstruction loss on raw event-image pixels — no JEPA target encoder, no EMA, no representation-collapse mode.** This is the safe fallback for event-camera SSL when JEPA + 1-bit-per-event-token has been resisting every anti-collapse fix.

**Pipeline:**

1. **Voxelize** each event clip to a 3-channel "event image" `(3, 128, 128)`:
   - `C0` = positive-polarity event count per pixel (normalized).
   - `C1` = latest-timestamp time-surface per pixel (normalized).
   - `C2` = negative-polarity event count per pixel (normalized).
2. **Patchify** with a 8×8 conv → 16×16 = 256 tokens of dim 192.
3. **Random mask 75% of patches** (MAE default).
4. **Encoder ViT-Tiny** (depth=6, dim=192, heads=6) on visible patches.
5. **Decoder ViT** (depth=4, dim=128) takes visible tokens + mask tokens at masked positions, predicts the original pixel values for masked patches.
6. **Loss = MSE** on masked patches only.

**Linear probe** on top of the frozen encoder gives the downstream signal.


## 1. Setup

In [ ]:
import os, sys, math, time, copy, ssl, glob
ssl._create_default_https_context = ssl._create_unverified_context

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0)

N_GPUS_REQUESTED = 4
N_GPUS = min(N_GPUS_REQUESTED, torch.cuda.device_count())
USE_DP = (DEVICE == "cuda") and (N_GPUS > 1)
print(f"GPUs visible = {torch.cuda.device_count()}  using = {N_GPUS}  DataParallel = {USE_DP}")

DATA_ROOT  = "../data/cifar10_dvs_omnibird"
TRAIN_FRAC = 0.8

class Config: pass
cfg = Config()
cfg.image_h        = 128
cfg.image_w        = 128
cfg.patch_size     = 8
cfg.n_patches      = (cfg.image_h // cfg.patch_size) * (cfg.image_w // cfg.patch_size)  # 256
cfg.mask_ratio     = 0.75
cfg.enc_dim        = 192
cfg.enc_depth      = 6
cfg.enc_heads      = 6
cfg.dec_dim        = 128
cfg.dec_depth      = 4
cfg.dec_heads      = 4
cfg.in_channels    = 3

cfg.epochs         = 100
cfg.batch_size     = 64
cfg.lr             = 1.5e-4 * (cfg.batch_size / 256)
cfg.weight_decay   = 0.05
cfg.warmup_epochs  = 5
cfg.probe_interval = 10
cfg.probe_epochs   = 2
cfg.log_every      = 25
cfg.ckpt_dir       = "./checkpoints_event_mae"
os.makedirs(cfg.ckpt_dir, exist_ok=True)
print(f"image {cfg.image_h}x{cfg.image_w}  patch {cfg.patch_size}  tokens {cfg.n_patches}  mask {cfg.mask_ratio:.0%}")


## 2. CIFAR10-DVS → 3-channel event image

In [ ]:
def events_to_image(ev, H=128, W=128):
    # ev: (N, 4) with cols x_norm, y_norm, t_norm, polarity_norm in [-1, 1].
    # Returns float32 (3, H, W) with channels [ON-count, time-surface, OFF-count],
    # each independently min-max normalized to [0, 1].
    if ev.shape[0] == 0:
        return np.zeros((3, H, W), dtype=np.float32)
    x = ((ev[:, 0] + 1.0) * 0.5 * (W - 1)).astype(np.int64)
    y = ((ev[:, 1] + 1.0) * 0.5 * (H - 1)).astype(np.int64)
    t = ev[:, 2]                            # in [-1, 1]
    p = ev[:, 3]
    x = np.clip(x, 0, W - 1); y = np.clip(y, 0, H - 1)

    img = np.zeros((3, H, W), dtype=np.float32)
    pos = p > 0; neg = p < 0
    np.add.at(img[0], (y[pos], x[pos]), 1.0)
    np.add.at(img[2], (y[neg], x[neg]), 1.0)
    # Time surface: latest t per pixel (sort by t ascending, assign — later overwrites earlier).
    order = np.argsort(t, kind="stable")
    img[1, y[order], x[order]] = (t[order] + 1.0) * 0.5  # [0, 1]

    # Per-channel max normalization
    for c in range(3):
        m = img[c].max()
        if m > 0: img[c] /= m
    return img


class CIFAR10DVSEventImages(Dataset):
    def __init__(self, root, H=128, W=128):
        self.root = root; self.H = H; self.W = W
        self.clips = sorted(glob.glob(os.path.join(root, "clip_*")))
        if not self.clips:
            raise RuntimeError(f"No clip_* in {root} (resolved: {os.path.abspath(root)})")
    def __len__(self): return len(self.clips)
    def __getitem__(self, idx):
        clip = self.clips[idx]
        ev = np.load(os.path.join(clip, "events_0.npy")).astype(np.float32)
        # Convert raw pixel coords / microseconds to [-1, 1].
        if ev.shape[0] == 0:
            ev_norm = np.zeros((1, 4), dtype=np.float32)
        else:
            x = ev[:, 0] / 127.0 * 2.0 - 1.0
            y = ev[:, 1] / 127.0 * 2.0 - 1.0
            t = ev[:, 2]; t = (t - t.min()) / max(t.max() - t.min(), 1.0) * 2.0 - 1.0
            p = ev[:, 3]
            if p.max() <= 1.0 and p.min() >= 0.0: p = p * 2.0 - 1.0
            ev_norm = np.stack([x, y, t, p], axis=1).astype(np.float32)
        img = events_to_image(ev_norm, self.H, self.W)
        with open(os.path.join(clip, "label_0.txt")) as f: label = int(f.read().strip())
        return torch.from_numpy(img), label


full = CIFAR10DVSEventImages(DATA_ROOT, cfg.image_h, cfg.image_w)
n = len(full); rng = np.random.RandomState(0)
perm = rng.permutation(n); n_train = int(n * TRAIN_FRAC)
train_idx, test_idx = perm[:n_train], perm[n_train:]

class _Subset(Dataset):
    def __init__(self, base, idx): self.base = base; self.idx = idx
    def __len__(self): return len(self.idx)
    def __getitem__(self, k): return self.base[int(self.idx[k])]

train_ds, test_ds = _Subset(full, train_idx), _Subset(full, test_idx)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False, num_workers=0)
print(f"train={len(train_ds)}  test={len(test_ds)}")


## 3. Visualize a few clips

In [ ]:
classes = ['airplane','car','bird','cat','deer','dog','frog','horse','ship','truck']
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for col in range(6):
    img, lab = train_ds[col]
    img_np = img.numpy()
    axes[0, col].imshow(img_np[0], cmap='Reds'); axes[0, col].set_title(f"ON  | {classes[lab]}")
    axes[1, col].imshow(img_np[1], cmap='viridis'); axes[1, col].set_title("time-surface")
    axes[2, col].imshow(img_np[2], cmap='Blues'); axes[2, col].set_title("OFF")
    for r in range(3): axes[r, col].axis('off')
plt.tight_layout(); plt.show()


## 4. ViT MAE

In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, in_ch=3, dim=192, patch=8):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, dim, kernel_size=patch, stride=patch)
    def forward(self, x):
        x = self.proj(x)                                 # (B, D, H/P, W/P)
        return x.flatten(2).transpose(1, 2)              # (B, N, D)


class Attention(nn.Module):
    def __init__(self, dim, n_heads=6):
        super().__init__()
        self.h = n_heads; self.dh = dim // n_heads
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.scale = self.dh ** -0.5
    def forward(self, x):
        B, N, D = x.shape
        qkv = self.qkv(x).view(B, N, 3, self.h, self.dh).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1) * self.scale).softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, N, D)
        return self.proj(out)


class MLP(nn.Module):
    def __init__(self, dim, mult=4):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim * mult)
        self.fc2 = nn.Linear(dim * mult, dim)
    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class Block(nn.Module):
    def __init__(self, dim, n_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim); self.attn = Attention(dim, n_heads)
        self.norm2 = nn.LayerNorm(dim); self.mlp  = MLP(dim)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


def sinusoidal_2d_pos_embed(dim, h, w):
    # Standard 2D sinusoidal position embedding (fixed, non-trainable).
    assert dim % 4 == 0
    omega = 1.0 / (10000 ** (torch.arange(0, dim // 4) / (dim // 4)))
    y = torch.arange(h).float().unsqueeze(1) * omega.unsqueeze(0)
    x = torch.arange(w).float().unsqueeze(1) * omega.unsqueeze(0)
    pe_x = torch.cat([torch.sin(x), torch.cos(x)], dim=-1)
    pe_y = torch.cat([torch.sin(y), torch.cos(y)], dim=-1)
    pe = torch.zeros(h, w, dim)
    pe[..., :dim//2] = pe_y.unsqueeze(1).expand(h, w, dim//2)
    pe[..., dim//2:] = pe_x.unsqueeze(0).expand(h, w, dim//2)
    return pe.reshape(h * w, dim)


class EventMAE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.patch_embed = PatchEmbed(cfg.in_channels, cfg.enc_dim, cfg.patch_size)
        n_h = cfg.image_h // cfg.patch_size
        n_w = cfg.image_w // cfg.patch_size
        self.register_buffer("enc_pos",
            sinusoidal_2d_pos_embed(cfg.enc_dim, n_h, n_w).unsqueeze(0), persistent=False)
        self.register_buffer("dec_pos",
            sinusoidal_2d_pos_embed(cfg.dec_dim, n_h, n_w).unsqueeze(0), persistent=False)
        self.enc_blocks = nn.ModuleList([Block(cfg.enc_dim, cfg.enc_heads) for _ in range(cfg.enc_depth)])
        self.enc_norm = nn.LayerNorm(cfg.enc_dim)

        self.enc_to_dec = nn.Linear(cfg.enc_dim, cfg.dec_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, cfg.dec_dim))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        self.dec_blocks = nn.ModuleList([Block(cfg.dec_dim, cfg.dec_heads) for _ in range(cfg.dec_depth)])
        self.dec_norm = nn.LayerNorm(cfg.dec_dim)
        self.dec_pred = nn.Linear(cfg.dec_dim, cfg.patch_size * cfg.patch_size * cfg.in_channels)

    @staticmethod
    def random_mask(B, N, mask_ratio, device):
        len_keep = int(N * (1 - mask_ratio))
        noise = torch.rand(B, N, device=device)
        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)
        ids_keep = ids_shuffle[:, :len_keep]
        mask = torch.ones(B, N, device=device)
        mask[:, :len_keep] = 0.0
        mask = torch.gather(mask, dim=1, index=ids_restore)
        return ids_keep, ids_restore, mask, len_keep

    def patchify(self, x):
        # x: (B, C, H, W) → (B, N, C*P*P) raw pixel patches.
        B, C, H, W = x.shape
        P = self.cfg.patch_size
        x = x.unfold(2, P, P).unfold(3, P, P)             # (B, C, H/P, W/P, P, P)
        x = x.permute(0, 2, 3, 1, 4, 5).contiguous()      # (B, H/P, W/P, C, P, P)
        return x.view(B, -1, C * P * P)

    def forward(self, imgs, mask_ratio=None):
        B = imgs.size(0)
        if mask_ratio is None: mask_ratio = self.cfg.mask_ratio
        x = self.patch_embed(imgs) + self.enc_pos
        N = x.size(1)
        ids_keep, ids_restore, mask, len_keep = self.random_mask(B, N, mask_ratio, x.device)
        x_keep = torch.gather(x, 1, ids_keep.unsqueeze(-1).expand(-1, -1, x.size(-1)))
        for blk in self.enc_blocks: x_keep = blk(x_keep)
        x_keep = self.enc_norm(x_keep)
        # Decoder: project + insert mask tokens at masked positions + add dec_pos
        x_dec = self.enc_to_dec(x_keep)
        mask_tok = self.mask_token.expand(B, N - len_keep, -1)
        x_full = torch.cat([x_dec, mask_tok], dim=1)
        x_full = torch.gather(x_full, 1, ids_restore.unsqueeze(-1).expand(-1, -1, x_full.size(-1)))
        x_full = x_full + self.dec_pos
        for blk in self.dec_blocks: x_full = blk(x_full)
        x_full = self.dec_norm(x_full)
        pred = self.dec_pred(x_full)                       # (B, N, C*P*P)

        # Target = original patch pixels (no normalization for now — channels are already [0,1]).
        target = self.patchify(imgs)
        # MSE on masked patches only
        loss = (pred - target) ** 2
        loss = loss.mean(dim=-1)                           # (B, N)
        loss = (loss * mask).sum() / mask.sum().clamp(min=1)
        return loss, pred, target, mask

    def encode_only(self, imgs):
        # For probe: full forward without masking, return per-patch features.
        x = self.patch_embed(imgs) + self.enc_pos
        for blk in self.enc_blocks: x = blk(x)
        return self.enc_norm(x)                            # (B, N, enc_dim)


model = EventMAE(cfg).to(DEVICE)
if USE_DP:
    model = nn.DataParallel(model, device_ids=list(range(N_GPUS)))

def _unwrap(m): return m.module if isinstance(m, nn.DataParallel) else m
n_params = sum(p.numel() for p in _unwrap(model).parameters())
print(f"EventMAE: {n_params/1e6:.2f} M params")


## 5. Optim + resume

In [ ]:
params = list(_unwrap(model).parameters())
optimizer = AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay)
total_steps = cfg.epochs * len(train_loader)
warmup_steps = cfg.warmup_epochs * len(train_loader)
def lr_lambda(step):
    if step < warmup_steps:
        return step / max(warmup_steps, 1)
    p = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return 0.5 * (1 + math.cos(math.pi * p))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

LAST = os.path.join(cfg.ckpt_dir, "event_mae_last.pt")
BEST = os.path.join(cfg.ckpt_dir, "event_mae_best.pt")
RESUME = True
history = {"loss": [], "probe_accs": []}
start_epoch, best_loss, global_step = 1, float("inf"), 0
if RESUME and os.path.exists(LAST):
    s = torch.load(LAST, map_location=DEVICE, weights_only=False)
    _unwrap(model).load_state_dict(s["model"])
    optimizer.load_state_dict(s["opt"]); scheduler.load_state_dict(s["sched"])
    history = s.get("history", history)
    global_step = s.get("global_step", 0); best_loss = s.get("best_loss", float("inf"))
    start_epoch = s["epoch"] + 1
    print(f"resumed @ ep {s['epoch']}, step {global_step}")
else:
    print("starting fresh.")


## 6. Linear probe

In [ ]:
class LinearProbe(nn.Module):
    def __init__(self, d, n=10):
        super().__init__()
        self.fc = nn.Linear(d, n)
    def forward(self, z): return self.fc(z)

def quick_probe(num_epochs=cfg.probe_epochs, lr=1e-3, wd=1e-4):
    enc = _unwrap(model)
    saved_rg = [p.requires_grad for p in enc.parameters()]
    for p in enc.parameters(): p.requires_grad_(False)
    enc.eval()
    probe = LinearProbe(cfg.enc_dim, 10).to(DEVICE)
    opt = AdamW(probe.parameters(), lr=lr, weight_decay=wd)
    ce  = nn.CrossEntropyLoss()
    def _z(imgs):
        with torch.no_grad():
            feat = enc.encode_only(imgs)            # (B, N, D)
        return feat.mean(dim=1)                     # (B, D)
    for _ in range(num_epochs):
        probe.train()
        for imgs, y in train_loader:
            imgs = imgs.to(DEVICE); y = y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            ce(probe(_z(imgs)), y).backward()
            opt.step()
    probe.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, y in test_loader:
            imgs = imgs.to(DEVICE); y = y.to(DEVICE)
            correct += (probe(_z(imgs)).argmax(-1) == y).sum().item(); total += y.size(0)
    for p, rg in zip(enc.parameters(), saved_rg): p.requires_grad_(rg)
    return correct / max(total, 1)


## 7. Training loop

In [ ]:
epoch = start_epoch - 1
try:
    for epoch in range(start_epoch, cfg.epochs + 1):
        model.train()
        epoch_loss, steps = 0.0, 0
        t0 = time.time()
        for imgs, _ in train_loader:
            imgs = imgs.to(DEVICE)
            loss, _, _, _ = model(imgs)
            if loss.dim() > 0: loss = loss.mean()   # in case DataParallel returns per-replica
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step(); scheduler.step()
            global_step += 1; epoch_loss += loss.item(); steps += 1
            if global_step % cfg.log_every == 0:
                print(f"[ep{epoch:02d} st{global_step:05d}]  recon_loss={loss.item():.4f}  "
                      f"lr={scheduler.get_last_lr()[0]:.1e}")
        avg = epoch_loss / max(steps, 1)
        history["loss"].append(avg)
        improved = avg < best_loss
        if improved: best_loss = avg
        state = {
            "epoch": epoch, "model": _unwrap(model).state_dict(),
            "opt": optimizer.state_dict(), "sched": scheduler.state_dict(),
            "global_step": global_step, "best_loss": best_loss, "history": history,
        }
        torch.save(state, LAST)
        if improved: torch.save(state, BEST)
        print(f"=== ep {epoch:03d}/{cfg.epochs}  recon_loss={avg:.4f}  {time.time()-t0:.1f}s"
              + ("  ★" if improved else ""))

        if cfg.probe_interval > 0 and epoch % cfg.probe_interval == 0:
            t_p = time.time()
            acc = quick_probe()
            history["probe_accs"].append((epoch, acc))
            print(f"  [probe ep {epoch:03d}]  test_acc = {acc:.4f}  ({time.time()-t_p:.1f}s)")
    print("\nDone.")
except KeyboardInterrupt:
    print(f"\nInterrupted at epoch {epoch}.  Checkpoints in {cfg.ckpt_dir}.")


## 8. Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
if history["loss"]:
    axes[0].plot(range(1, len(history["loss"])+1), history["loss"], lw=1.5)
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("MSE"); axes[0].set_title("MAE reconstruction loss"); axes[0].grid(alpha=0.3)
if history.get("probe_accs"):
    eps, accs = zip(*history["probe_accs"])
    axes[1].plot(eps, [a*100 for a in accs], 'o-', color='C3')
    axes[1].set_ylim(0, 100); axes[1].set_xlabel("epoch"); axes[1].set_ylabel("probe acc (%)")
    axes[1].set_title(f"linear probe (best = {max(accs)*100:.2f}%)"); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 9. Reconstruction visualization

In [ ]:
# Visualize a few masked → reconstructed images
model.eval()
with torch.no_grad():
    imgs, _ = next(iter(test_loader))
    imgs = imgs[:8].to(DEVICE)
    loss, pred, target, mask = _unwrap(model)(imgs)
    print(f"recon MSE on test batch = {loss.item():.4f}")

# Unpatchify: pred and target are (B, N, C*P*P)
P = cfg.patch_size; C = cfg.in_channels
nH = cfg.image_h // P; nW = cfg.image_w // P
def unpatchify(p):
    B, N, _ = p.shape
    p = p.reshape(B, nH, nW, C, P, P).permute(0, 3, 1, 4, 2, 5)
    return p.reshape(B, C, cfg.image_h, cfg.image_w)
target_img = unpatchify(target).cpu().numpy()
pred_img   = unpatchify(pred).cpu().numpy()
mask_img   = mask.unsqueeze(-1).expand(-1, -1, C*P*P)
masked_in  = unpatchify(target * (1 - mask_img)).cpu().numpy()

fig, axes = plt.subplots(3, 8, figsize=(20, 8))
for i in range(8):
    axes[0, i].imshow(target_img[i, 0], cmap='Reds');  axes[0, i].set_title("orig ON"); axes[0, i].axis('off')
    axes[1, i].imshow(masked_in [i, 0], cmap='Reds');  axes[1, i].set_title("masked");  axes[1, i].axis('off')
    axes[2, i].imshow(pred_img  [i, 0], cmap='Reds');  axes[2, i].set_title("recon");   axes[2, i].axis('off')
plt.tight_layout(); plt.show()
